# Examples of the fractiona-derivative method


## Poisson-Gamma Mixture (Negative Binomial)

This notebook demonstrates the use of `evidence()` to compute the marginal likelihood (evidence) for a Poisson likelihood with a Gamma prior. The Poisson-Gamma mixture is equivalent to a Negative Binomial distribution, which we use as a ground truth for validation.

We will:
- Define a small dataset.
- Compute the evidence using:
  - `method='symbolic'` (numeric evaluation)
  - `method='bell'` (Bell polynomial)
  - `method='jax'` (JAX via `jet`)
- Compare results with the Negative Binomial log‑PMF.
- Also show the symbolic expression for the evidence.

### Setup

First, import the necessary modules and functions.

In [1]:
import pandas as pd
import numpy as np
import sympy as sp
import math
from MGFderivative_class import MGFDerivative

### Define data and parameters

We'll use a simple dataset of counts and a Gamma prior.

In [2]:
# Data: observed counts
data = pd.DataFrame({'counts': [1, 2, 3, 0, 1, 2]})

# Prior parameters for Gamma(alpha, beta)
alpha_prior = 2.0
beta_prior = 3.0
params = {'alpha': alpha_prior, 'beta': beta_prior}

# Exposure scale (default is 1 for all observations)
scale = 1.0

### Compute evidence with different methods

We'll compute the log‑evidence (log of the marginal likelihood) using the three methods. All return (log_abs, sign) because log=True by default.

In [3]:
# Symbolic method (numeric evaluation)
deriv_sym = MGFDerivative(prior='gamma', data=data, method='symbolic',
                          params=params, log=True, scale=scale)
log_ev_sym, sign_sym = deriv_sym.evidence()

# Bell polynomial method
deriv_bell = MGFDerivative(prior='gamma', data=data, method='bell',
                           params=params, log=True, scale=scale)
log_ev_bell, sign_bell = deriv_bell.evidence()

# JAX method
deriv_jax = MGFDerivative(prior='gamma', data=data, method='jax',
                          params=params, log=True, scale=scale)
log_ev_jax, sign_jax = deriv_jax.evidence()

print(f"Symbolic: log|evidence| = {log_ev_sym:.6f}, sign = {sign_sym}")
print(f"Bell:     log|evidence| = {log_ev_bell:.6f}, sign = {sign_bell}")
print(f"JAX:      log|evidence| = {log_ev_jax:.6f}, sign = {sign_jax}")

🔬 Testing symbolic derivative of order 2 (target order: 9)...
   ✅ Test succeeded in 0.002s, complexity=3
Decision: Symbolic
Symbolic: log|evidence| = -10.045887, sign = 1
Bell:     log|evidence| = -10.045887, sign = 1
JAX:      log|evidence| = -10.045887, sign = 1


For the Poisson-Gamma model:

- Likelihood: $ Y_i \mid \theta \sim \text{Poisson}(s_i \theta) $
- Prior: $ \theta \sim \text{Gamma}(\alpha, \beta) $

The joint marginal likelihood (evidence) is:

$$
p(\mathbf{y}) = \frac{\beta^\alpha}{\Gamma(\alpha)} \,
\prod_i \frac{s_i^{y_i}}{y_i!} \,
\frac{\Gamma(\alpha + a)}{(\beta + b)^{\alpha + a}},
$$

where  
$ a = \sum_i y_i $,  
$ b = \sum_i s_i $.

Its log is:

$$
\log p(\mathbf{y}) = \underbrace{\sum_i (y_i \log s_i - \log y_i!)}_{\log c}
+ \log \Gamma(\alpha + a) - \log \Gamma(\alpha)
+ \alpha \log \beta - (\alpha + a)\log(\beta + b).
$$

This is exactly what `evidence()` computes: `log_c + log_derivative`, where the derivative is  
$\frac{\Gamma(\alpha+a)}{\Gamma(\alpha)} \beta^\alpha (\beta+b)^{-(\alpha+a)}$.

In [4]:
# Direct analytical joint likelihood
def joint_poisson_gamma_logpmf(y, s, alpha, beta):
    import math
    a = sum(y)
    b = sum(s)
    log_c = sum(y_i * math.log(s_i) - math.lgamma(y_i + 1) for y_i, s_i in zip(y, s))
    log_deriv = (math.lgamma(alpha + a) - math.lgamma(alpha)
                 + alpha * math.log(beta)
                 - (alpha + a) * math.log(beta + b))
    return log_c + log_deriv

log_ev_direct = joint_poisson_gamma_logpmf(
    data['counts'].values,
    [scale] * len(data),
    alpha_prior,
    beta_prior
)

print(f"Direct analytical log-evidence: {log_ev_direct:.6f}")
print(f"Difference from evidence: {log_ev_sym - log_ev_direct:.2e}")
print(f"Difference from evidence: {log_ev_bell - log_ev_direct:.2e}")
print(f"Difference from evidence: {log_ev_jax - log_ev_direct:.2e}")

Direct analytical log-evidence: -10.045887
Difference from evidence: 3.55e-15
Difference from evidence: 3.55e-15
Difference from evidence: 7.11e-15


Symbolic expression (un‑evaluated)

We can also obtain the symbolic expression for the evidence, which is useful for analytical manipulation.

In [6]:
# Symbolic expression (un‑evaluated)
deriv_sym = MGFDerivative(
    prior='gamma',
    data=data,
    method='symbolic',
    params=None,      # no numeric parameters -> symbolic
    simplify=True,    # optional
    scale=scale
)
expr_sym = deriv_sym.evidence()

print("Symbolic evidence expression (simplified):")
sp.pprint(expr_sym, use_unicode=False)

Symbolic evidence expression (simplified):
                                                                               >
                                                                               >
                alpha                                                          >
      /  beta  \      /     8           7            6             5           >
alpha*|--------|     *\alpha  + 36*alpha  + 546*alpha  + 4536*alpha  + 22449*a >
      \beta - t/                                                               >
                                                                               >
                                                                               >
------------------------------------------------------------------------------ >
                                                                               >
                                                                         (beta >

>                                                                